# 07 — Reviewed YouTube training batch v4 to pose landmarks

This notebook converts four newly reviewed, single-person workout videos into the same normalized landmark format used by the cALorie TCN.

It:

1. downloads four public source videos temporarily,
2. downloads the human-reviewed contiguous timelines from the project branch,
3. samples each video at 15 FPS and extracts 17 normalized MediaPipe landmarks,
4. cuts the timelines into 212 labelled segments,
5. preserves one `group_id` per source video to prevent leakage,
6. audits detection quality and downloads `calorie_youtube_training_landmarks_v4.zip`.

Use a **CPU runtime**. MediaPipe Pose runs on the CPU in this notebook; attaching a T4 wastes paid compute units without speeding this pipeline enough to justify it.

The two-person Growingannanas and HASfit candidates were rejected before annotation. The locked Heather Robertson no-repeat video is also excluded and remains an unseen external test.


In [ ]:
%pip install -q yt-dlp mediapipe==0.10.35 opencv-python-headless pandas==2.2.3 tqdm


In [ ]:
from pathlib import Path
import csv
import hashlib
import shutil
import subprocess
import urllib.request

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from google.colab import files

TARGET_FPS = 15.0
WORK_ROOT = Path("/content/calorie_youtube_v4_work")
VIDEO_ROOT = WORK_ROOT / "videos"
LABEL_ROOT = WORK_ROOT / "labels"
OUTPUT_ROOT = Path("/content/calorie_youtube_training_landmarks_v4")
for path in (VIDEO_ROOT, LABEL_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

BRANCH = "feat/tcn-rebuild"
RAW_BASE = f"https://raw.githubusercontent.com/bariscelikk1/cALorie/{BRANCH}/ml/data/youtube_mixed_workouts_v1"
SOURCES = [
    ("UItWltVZZmE", "pamela_reif__UItWltVZZmE"),
    ("Gzp8IA2vW_Q", "heather_robertson_circuit__Gzp8IA2vW_Q"),
    ("fegQSC-nUtg", "caroline_girvan__fegQSC-nUtg"),
    ("2Z9g-AZinUc", "madfit_small_space__2Z9g-AZinUc"),
]
print("Output:", OUTPUT_ROOT)


## Download source videos and reviewed labels

Format `134` is 640×360 H.264 video. That resolution is enough for pose landmarks and decodes reliably with OpenCV. Raw videos stay only in this temporary Colab runtime.


In [ ]:
for video_id, stem in SOURCES:
    video_path = VIDEO_ROOT / f"{stem}.mp4"
    label_path = LABEL_ROOT / f"{stem}.labels.csv"
    if not video_path.exists():
        subprocess.run([
            "yt-dlp", "--no-playlist", "-f", "134", "-o", str(video_path),
            f"https://www.youtube.com/watch?v={video_id}",
        ], check=True)
    urllib.request.urlretrieve(
        f"{RAW_BASE}/annotations/reviewed/{stem}.labels.csv", label_path
    )
    print(video_id, f"{video_path.stat().st_size / 1e6:.1f} MB", label_path.name)


In [ ]:
MODEL_PATH = WORK_ROOT / "pose_landmarker_full.task"
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_full/float16/latest/pose_landmarker_full.task"
)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Pose model ready:", MODEL_PATH.exists())


## Pose extraction

MediaPipe returns 33 points. We keep the same 17-point layout used by the existing TCN. Each pose is centred on the hips and divided by shoulder width or torso length, which reduces camera-distance and body-position differences. Missing detections are interpolated and every segment records its detection rate.


In [ ]:
KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    shoulder_width = np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])
    torso_length = np.linalg.norm(shoulder_center - hip_center)
    scale = max(float(shoulder_width), float(torso_length), 1e-6)
    xyz = (xyz - hip_center) / scale
    return np.concatenate([xyz, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction="both").fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def extract_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")
    source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    next_sample = 0.0
    frame_index = 0
    poses, timestamps, detected = [], [], []
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            if time_s + 1e-9 < next_sample:
                continue
            next_sample += 1.0 / TARGET_FPS
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, int(round(time_s * 1000)))
            if result.pose_landmarks:
                points = result.pose_landmarks[0]
                pose = np.asarray([
                    [points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0.0)]
                    for i in KEEP_17
                ], dtype=np.float32)
                poses.append(normalize_pose(pose))
                detected.append(True)
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
                detected.append(False)
            timestamps.append(time_s)
    cap.release()
    if not poses:
        raise RuntimeError(f"No frames decoded from {video_path}")
    return (
        fill_missing(np.stack(poses)),
        np.asarray(timestamps, dtype=np.float32),
        np.asarray(detected, dtype=bool),
    )


def make_clip_id(video_id, start, end, label):
    raw = f"{video_id}|{start:.3f}|{end:.3f}|{label}".encode()
    return hashlib.sha256(raw).hexdigest()[:16]


## Convert reviewed timelines into training clips

Rest, previews, intros and unsupported exercises become `unknown`. This is intentional: it teaches the model to reject motion outside its supported classes. Every segment from the same source gets the same `group_id`.


In [ ]:
manifest_rows = []
for video_id, stem in tqdm(SOURCES, desc="Source videos"):
    video_path = VIDEO_ROOT / f"{stem}.mp4"
    label_path = LABEL_ROOT / f"{stem}.labels.csv"
    labels = pd.read_csv(label_path)
    sequence, timestamps, detected = extract_video(video_path)
    print(f"{video_id}: {len(sequence)} frames, detection={detected.mean():.3f}")

    for row in labels.itertuples(index=False):
        start, end = float(row.start_s), float(row.end_s)
        indices = np.flatnonzero((timestamps >= start) & (timestamps < end))
        if not len(indices):
            continue
        canonical = row.current_tcn_label
        identifier = make_clip_id(video_id, start, end, canonical)
        class_dir = OUTPUT_ROOT / canonical
        class_dir.mkdir(parents=True, exist_ok=True)
        output_path = class_dir / f"{identifier}.npz"
        np.savez_compressed(
            output_path,
            landmarks=sequence[indices].astype(np.float32),
            timestamps_s=(timestamps[indices] - start).astype(np.float32),
        )
        rate = float(detected[indices].mean())
        manifest_rows.append({
            "clip_id": identifier,
            "source_dataset": "youtube_training_workouts_v4",
            "source_class": row.fine_label,
            "canonical_class": canonical,
            "original_path": f"{video_path.name}#{start:.1f}-{end:.1f}",
            "output_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "fingerprint": video_id,
            "status": "ok" if rate >= 0.5 else "low_detection",
            "source_fps": TARGET_FPS,
            "sampled_frames": len(indices),
            "detection_rate": rate,
            "duration_s": end - start,
            "group_id": f"youtube:{video_id}",
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
print("Saved segments:", len(manifest))
display(manifest.groupby("canonical_class").agg(
    clips=("clip_id", "count"),
    frames=("sampled_frames", "sum"),
    detection_mean=("detection_rate", "mean"),
    detection_min=("detection_rate", "min"),
).round(3))


## Audit and download

The assertions stop the notebook before packaging an incomplete run. Low-detection segments remain visible in the report and should be reviewed before training.


In [ ]:
npz_files = list(OUTPUT_ROOT.rglob("*.npz"))
print("NPZ files:", len(npz_files))
print("Status counts:")
print(manifest["status"].value_counts())
print("Groups:", sorted(manifest["group_id"].unique()))

assert len(manifest) == 212, f"Expected 212 reviewed segments, found {len(manifest)}"
assert len(npz_files) == len(manifest), "Manifest/NPZ count mismatch"
assert manifest["group_id"].nunique() == 4, "Expected four source-video groups"
assert manifest["output_path"].map(lambda p: (OUTPUT_ROOT / p).exists()).all()

zip_path = Path(shutil.make_archive(
    "/content/calorie_youtube_training_landmarks_v4", "zip", root_dir=OUTPUT_ROOT
))
print("ZIP:", zip_path)
print("ZIP size MB:", round(zip_path.stat().st_size / 1e6, 2))
files.download(str(zip_path))
